<a href="https://colab.research.google.com/github/gmauricio-toledo/NLP-LCC/blob/main/Tareas%20y%20Actividades/SOL-TopicModeling_NoticiasUNISON.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1>Actividad en clase: Topic Modeling</h1>

In [ ]:
!gdown 1mE4dVNVqS1Xt28xz7ngHZrtGM0WTSrQ0

In [ ]:
import pandas as pd

df = pd.read_csv('noticias_unison.csv')
df = df[['titulo','texto']]
df

Entrena un modelo de topic modeling para encontrar los tópicos principales en las noticias del sitio web de la unison.

Utiliza dos estrategias:
* Clustering con representaciones doc2vec
* LSA

En ambos casos describe cada uno de los tópicos que hayas encontrado. En el primer caso, usa una nube de palabras por cada tópico. En el segundo, infiere el tópico a partir de las principales palabras de cada tópico.

## Clustering

In [ ]:
!pip install gensim

In [ ]:
from string import punctuation
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt_tab',quiet=True)
nltk.download('stopwords',quiet=True)

stopwords = nltk.corpus.stopwords.words('spanish')
stopwords.extend(['unison','universidad','sonora'])

punctuation += '¿¡”“'

df['all_text'] = df['titulo'] + ' ' + df['texto']
docs = df['all_text'].values

tokenized_docs = [[x.lower() for x in word_tokenize(doc) if x.lower() not in stopwords and x not in punctuation] for doc in docs]

print(docs[0])
print(tokenized_docs[0])

Etiquetamos los documentos para doc2vec

In [ ]:
tagged_docs = [TaggedDocument(words=d, tags=[str(i)]) for i, d in enumerate(tokenized_docs)]
tagged_docs[:3]

Entrenamos el modelo

In [ ]:
model = Doc2Vec(tagged_docs, vector_size=70, window=3, min_count=1, workers=4)
model.build_vocab(tagged_docs)
model.train(tagged_docs, total_examples=model.corpus_count, epochs=7)

Vectorizamos los documentos

In [ ]:
# Si son los documentos con los que entrenamos sólo los extraemos:
vectors = model.dv.vectors
vectors[:3,:5]

# Alternativamente, podemos inferir los vectores (principalmente en caso de que no sean los documentos con los que entrenamos)_
# vectors = [model.infer_vector(doc) for doc in tokenized_docs]
# vectors

Entrenamos el modelo de clustering

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

num_clusters = 6
clustering = AgglomerativeClustering(n_clusters=num_clusters)
clustering.fit(vectors)
labels = clustering.labels_ # Los tópicos

Visualizamos los tópicos

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import numpy as np

docs_per_cluster = []

for k in range(num_clusters):
    idxs = np.where(labels==k)[0]
    tok_docs_per_cluster = [tokenized_docs[j] for j in idxs]
    docs_per_cluster.append([' '.join(x) for x in tok_docs_per_cluster])

plt.figure(figsize=(10,10))
for k in range(num_clusters):
    text = ' '.join(docs_per_cluster[k])
    wordcloud = WordCloud()
    wc = wordcloud.generate(text)
    plt.subplot(3,2,k+1)
    plt.axis('off')
    plt.imshow(wc)
plt.show()

Evaluamos el **clustering**

In [ ]:
print(f"Silhouette score: {silhouette_score(vectors, labels)}")

## LSA

Obtenemos primero la matriz de conteos BOW

In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(stop_words=stopwords, max_features=2500)
X_bow = cv.fit_transform(docs).todense()
X_bow = np.array(X_bow)
X_bow.shape

Entrenamos el modelo de LSA y realizamos las predicciones (proyecciones)

In [ ]:
lsa_model = TruncatedSVD(n_components=num_clusters)
lsa_model.fit(X_bow)

doc_projections = lsa_model.transform(X_bow)
word_projections = lsa_model.components_.transpose()

print(f"Doc projections shape: {doc_projections.shape}")
print(f"Word projections shape: {word_projections.shape}")

Obtenemos las palabras más representativas de cada tópico

In [ ]:
# Obtiene el vocabulario del vectorizer como array (cada índice corresponde a una columna de la matriz BOW)
terms = cv.get_feature_names_out()

num_top_words = 11

# Itera sobre cada tópico (fila de components_), i es el número de tópico, comp es el vector de pesos
for i, comp in enumerate(lsa_model.components_):
    # Empareja cada palabra con su peso en este tópico → lista de tuplas (palabra, peso)
    terms_comp = zip(terms, comp)
    # Ordena por peso descendente y se queda con las 'num_top_words' palabras más representativas del tópico
    sorted_terms = sorted(terms_comp, key= lambda x:x[1], reverse=True)[:num_top_words]
    print(f"Topic {str(i)}: ")
    # Imprime solo el nombre de la palabra (t[0]) de cada tupla
    print(', '.join([t[0] for t in sorted_terms]))
    # for t in sorted_terms:
    #     print(t[0],end=', ')
    print('\n'+75*'=')

In [ ]:
#@title Implementación coherencia `u_mass`

import math

def get_umass_score(dt_matrix, i, j):
    # Binariza la matriz: 1 si el término aparece en el documento, 0 si no
    zo_matrix = (dt_matrix > 0).astype(int)

    # Extrae la columna del término i y del término j
    col_i, col_j = zo_matrix[:, i], zo_matrix[:, j]

    # Suma ambas columnas: valor 2 indica que ambos términos aparecen en el mismo doc
    col_ij = col_i + col_j
    col_ij = (col_ij == 2).astype(int)  # 1 donde co-ocurren, 0 donde no

    # D(i) = nº de documentos que contienen el término i
    # D(i,j) = nº de documentos que contienen ambos términos a la vez
    Di, Dij = col_i.sum(), col_ij.sum()

    # Fórmula U-Mass: log( (D(i,j) + 1) / D(i) )
    # +1 en el numerador evita log(0) cuando no hay co-ocurrencias
    return math.log((Dij + 1) / Di)


def get_topic_coherence(dt_matrix, topic, n_top_words):
    # Empareja cada peso del tópico con su índice de término
    indexed_topic = zip(topic, range(0, len(topic)))

    # Ordena por peso descendente y toma las n_top_words palabras más relevantes
    topic_top = sorted(indexed_topic, key=lambda x: 1 - x[0])[0:n_top_words]

    coherence = 0
    # Itera sobre todos los pares (i, j) con i < j entre las top words
    for j_index in range(0, len(topic_top)):
        for i_index in range(0, j_index):  # solo pares sin repetir
            i = topic_top[i_index][1]  # índice real del término i en la matriz
            j = topic_top[j_index][1]  # índice real del término j en la matriz

            # Acumula el score U-Mass del par (i, j)
            coherence += get_umass_score(dt_matrix, i, j)
    return coherence


def get_average_topic_coherence(dt_matrix, topics, n_top_words):
    total_coherence = 0

    # Suma la coherencia de cada tópico
    for i in range(0, len(topics)):
        total_coherence += get_topic_coherence(dt_matrix, topics[i], n_top_words)

    # Devuelve el promedio entre todos los tópicos
    return total_coherence / len(topics)

Evaluamos la coherencia

In [ ]:
get_average_topic_coherence(X_bow, lsa_model.components_, 10)

Obtenemos los tópicos

In [ ]:
lsa_labels = np.argmax(doc_projections,axis=1)

print(lsa_labels.shape)

docs_per_cluster = []

for k in range(num_clusters):
    idxs = np.where(labels==k)[0]
    tok_docs_per_cluster = [tokenized_docs[j] for j in idxs]
    docs_per_cluster.append([' '.join(x) for x in tok_docs_per_cluster])

plt.figure(figsize=(10,10))
for k in range(num_clusters):
    text = ' '.join(docs_per_cluster[k])
    wordcloud = WordCloud()
    wc = wordcloud.generate(text)
    plt.subplot(3,2,k+1)
    plt.axis('off')
    plt.imshow(wc)
plt.show()

¿Dejamos `estudiante`?